# 🎭 Dynamic Emotion Engine MVP — Qwen3-TTS

This notebook demonstrates how to use the Qwen3-TTS VoiceDesign model to create a reactive dialogue system for a game NPC. We will generate variations of a single line across multiple emotions and intensity levels, forming a matrix of audio assets that a game engine can dynamically select based on the game state.

In [ ]:
# Install necessary packages
!pip install -q qwen-tts soundfile

In [ ]:
import os
import gc
import torch
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

def to_wav(result, default_sr=24000):
    """Normalize any Qwen3-TTS generate_* return into (waveform, sample_rate)."""
    audio, sr = result if isinstance(result, tuple) else (result, default_sr)
    if isinstance(audio, (list, tuple)):
        audio = audio[0]
    if hasattr(audio, "cpu"):
        audio = audio.cpu().numpy()
    return audio, sr

OUTPUT_DIR = "/content/emotion_engine"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

In [ ]:
CHARACTER = {
    "name": "Captain Voss",
    "base_prompt": "A seasoned starship captain — authoritative, experienced, controlled, mid-50s with gravitas"
}

EMOTIONS = {
    "neutral":    {"modifier": "calm, measured, professional delivery", "intensity_notes": {"low": "very flat", "medium": "composed", "high": "firm"}},
    "happy":      {"modifier": "warm, pleased, slight smile in the voice", "intensity_notes": {"low": "content", "medium": "genuinely pleased", "high": "openly joyful, rare for this character"}},
    "angry":      {"modifier": "barely controlled fury, clipped words", "intensity_notes": {"low": "irritated", "medium": "clearly angry", "high": "shouting, barely restrained"}},
    "sad":        {"modifier": "heavy, subdued, emotional weight", "intensity_notes": {"low": "somber", "medium": "genuinely grieving", "high": "breaking voice, grief"}},
    "fearful":    {"modifier": "tense, slightly rushed, controlled panic", "intensity_notes": {"low": "uneasy", "medium": "clearly scared", "high": "barely keeping it together"}},
    "surprised":  {"modifier": "caught off guard, momentarily dropped composure", "intensity_notes": {"low": "mildly surprised", "medium": "genuinely startled", "high": "shocked, voice breaks briefly"}},
    "disgusted":  {"modifier": "cold contempt, slight curl in tone", "intensity_notes": {"low": "mild disdain", "medium": "clear disgust", "high": "barely tolerating speaking"}},
    "triumphant": {"modifier": "proud, expansive, victory in voice", "intensity_notes": {"low": "satisfied", "medium": "proud", "high": "exultant, rare for this character"}}
}

INTENSITIES = ["low", "medium", "high"]

TEST_SENTENCE = "All hands — report to your stations immediately."
print("Config loaded successfully.")

In [ ]:
# Clear memory just in case
gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
print(f"Loading model {model_id}...")
model = Qwen3TTSModel.from_pretrained(
    model_id,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)
print("Model loaded successfully!")

In [ ]:
total_clips = len(EMOTIONS) * len(INTENSITIES)
counter = 1

for emotion, emotion_data in EMOTIONS.items():
    for intensity in INTENSITIES:
        instruct = f"{CHARACTER['base_prompt']}, {emotion_data['modifier']}, {emotion_data['intensity_notes'][intensity]} intensity"
        
        print(f"\nGenerated {counter}/{total_clips}: {emotion}_{intensity}")
        print(f"Instruct: {instruct}")
        
        audio_array = model.generate_voice_design(
            text=TEST_SENTENCE,
            language="English",
            instruct=instruct
        )
        
        filename = f"emotion_{emotion}_{intensity}.wav"
        filepath = os.path.join(OUTPUT_DIR, filename)
        audio_array, sr = to_wav(audio_array)
        sf.write(filepath, audio_array, sr)
        
        print(f"[EMOTION: {emotion.upper()} | INTENSITY: {intensity.upper()}]")
        display(Audio(filepath))
        counter += 1

In [ ]:
print("\n=== Emotion Comparison: ANGRY ===")
for intensity in INTENSITIES:
    filepath = os.path.join(OUTPUT_DIR, f"emotion_angry_{intensity}.wav")
    print(f"Angry | Intensity: {intensity.upper()}")
    if os.path.exists(filepath):
        display(Audio(filepath))
    else:
        print("File not found.")

In [ ]:
print(f"EMOTION ENGINE GRID — {CHARACTER['name']}")
print("┌──────────────┬──────────────────────────────┬────────────────────────────────┬──────────────────────────────────┐")
print("│ Emotion      │ Low                          │ Medium                         │ High                             │")
print("├──────────────┼──────────────────────────────┼────────────────────────────────┼──────────────────────────────────┤")
for emotion in EMOTIONS.keys():
    row = f"│ {emotion:<12} "
    for intensity in INTENSITIES:
        filename = f"emotion_{emotion}_{intensity}.wav"
        row += f"│ {filename:<28} "
    row += "│"
    print(row)
print("└──────────────┴──────────────────────────────┴────────────────────────────────┴──────────────────────────────────┘")

In [ ]:
print("\n=== Reactive Dialogue Demo Scene ===")
print("Scene: The ship takes heavy damage (angry/high) -> systems stabilize (neutral/medium) -> crew cheers (happy/medium) -> enemy reinforcements arrive (fearful/medium)")

scene_sequence = [
    "emotion_angry_high.wav",
    "emotion_neutral_medium.wav",
    "emotion_happy_medium.wav",
    "emotion_fearful_medium.wav"
]

combined_audio = []
sample_rate = 24000
silence_1s = np.zeros(sample_rate, dtype=np.float32)

for fname in scene_sequence:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        audio_data, sr = sf.read(fpath)
        combined_audio.extend(audio_data)
        combined_audio.extend(silence_1s)

combined_audio = np.array(combined_audio, dtype=np.float32)
scene_path = os.path.join(OUTPUT_DIR, "emotion_reactive_scene.wav")
sf.write(scene_path, combined_audio, sample_rate)

display(Audio(scene_path))

In [ ]:
import shutil

# Zip the output directory for easy download
shutil.make_archive("/content/emotion_engine_assets", 'zip', OUTPUT_DIR)
print("Assets zipped successfully to /content/emotion_engine_assets.zip")

from google.colab import files
try:
    files.download("/content/emotion_engine_assets.zip")
except:
    print("File download is only available in a live Colab environment.")